# 1. Trazabilidad de correcciones

La tabla reúne cada observación con la acción realizada, su ubicación en este libro y la evidencia.
Incluye tres grupos: las observaciones del profesor al Entregable 1 (atendidas en el Entregable 2 y
retomadas aquí porque el modelo original aprovecha la dependencia espacial), las observaciones del
profesor al Entregable 2 y los problemas que el propio equipo encontró en el Entregable 2 al
preparar este entregable. Una corrección que cambia un resultado obliga a recalcular todo lo que
depende de él; la última columna dice qué se recalculó.

In [1]:
import sys
import warnings

import pandas as pd
from IPython.display import Markdown, display

sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 400)

In [2]:
trazabilidad = pd.read_csv("../data/resultados/trazabilidad.csv")
display(trazabilidad)

,Origen,Observación,Acción realizada,Ubicación en el libro,Evidencia,Resultados recalculados
0,"Profesor, Entregable 1",Atender la dependencia espacial que quedó en los residuos.,"En el Entregable 2 se diagnosticó por bloque (I de Moran y LISA con control de falsos descubrimientos), se descontaron los choques comunes departamento × semana y se añadieron variables de vecindad y la codificación del municipio. En el Entregable 3, el modelo original incorpora una codificación Bayes empírica jerárquica corredor → municipio → departamento.","Entregable 2, cap. 2; aquí, secciones 5.2, 5.3 (proposición 4) y 5.6 (ablación sin_jerarquico)","I de Moran de XGBoost de 0,053 a 0,043; aporte de la codificación jerárquica en la tabla de ablación",Sin cambios en los resultados del Entregable 2
1,"Profesor, Entregable 1",Revisar la brecha negativa de la curva de aprendizaje.,Era un error de procedimiento: learning_curve(shuffle=False) tomaba las filas en el orden del panel (centrales en orden alfabético). Se repitió con submuestras aleatorias.,"Entregable 2, sección 2.2","Figura de curvas de aprendizaje corregidas (brecha ≈ 0 en la logística, positiva y decreciente en XGBoost)",Curvas de aprendizaje
2,"Profesor, Entregable 2",Pendiente: la retroalimentación del Entregable 2 aún no se ha recibido.,Se agregará una fila por observación cuando llegue la retroalimentación.,—,—,—
3,Revisión propia,Empate exacto en AUC entre k-NN con votos ponderados y k-NN sin balanceo; el desempate quedaba al orden alfabético.,Regla de selección con desempate por Brier medio del bucle externo.,Sección 3.1 y tabla de selección,"seleccion_mejores.csv: k-NN sin balanceo (Brier 0,1661) frente a votos ponderados (0,1869)","Diagrama CD, pruebas del capítulo 6 y tablero"
4,Revisión propia,El umbral de decisión de la SVM se eligió en la escala del margen y su artefacto entrega probabilidades.,El umbral se transforma con el escalado de Platt del artefacto (transformación monótona: la decisión no cambia).,Sección 3.2,metadata.json de la SVM,Métricas de umbral de la SVM en 2025
5,Revisión propia,"Random Forest con varios hilos suma las probabilidades de los árboles en un orden variable (diferencias de 2,2e-16) y la prueba de ida y vuelta fallaba.",Todos los artefactos se serializan con n_jobs = 1 (Render ofrece un núcleo).,Sección 3.3,Prueba de ida y vuelta exacta en los 8 artefactos,Tiempos de inferencia
6,Revisión propia,"La columna «Cliff pareado» del Entregable 2 era la dominancia de las diferencias pareadas, no el delta de Cliff clásico.","Se reportan por separado el delta de Cliff (dos muestras) y la dominancia pareada, con su definición.",Sección 6.4,Tabla de tamaños del efecto,Tamaños del efecto
7,Revisión propia,"SMOTE y ADASYN se aplicaron sobre variables one-hot, lo que crea valores intermedios en las categorías.","Se documenta como limitación. Entre las siete combinaciones seleccionadas solo afecta a Naive Bayes (SMOTE), que es el modelo con menor AUC externo; su efecto sobre la comparación con el modelo original es nulo porque este no usa remuestreo.",Sección 8.3,—,Sin cambios


## 1.1 Cambios en los resultados

* **Mejor combinación de k-NN.** En el Entregable 2, k-NN con votos ponderados por clase y k-NN sin
  balanceo empataron exactamente en AUC externo (0,7751), porque la ponderación es una
  transformación monótona de la fracción de vecinos positivos y no cambia el orden de los puntajes.
  El desempate quedaba al orden alfabético. Ahora se desempata con el Brier medio, que sí distingue
  la calibración: gana k-NN sin balanceo (Brier 0,1661 frente a 0,1869). El ranking de los siete
  modelos no cambia, pero el diagrama de diferencia crítica, las pruebas de hipótesis y el tablero
  usan esta combinación.
* **Universo de la comparación estadística.** Los 140 modelos del Entregable 2 se comparaban entre
  sí; ahora el universo de las pruebas del capítulo 6 son las siete mejores combinaciones más el
  modelo original, sobre los mismos 48 meses del bucle externo y la misma prueba 2025.
* **Métricas de 2025 con todo el entrenamiento.** El Entregable 2 reentrenó con las 622 mil filas
  cinco de los siete modelos; aquí se reentrenan los siete (k-NN con su muestra de 40 mil filas, por
  la razón que explica el capítulo 3) y se reportan sus métricas completas.